# Phytoplankton event detection (using chl-a anamolies) from PACE OCI data products

**POC/Reviewers:** Matthew Kehrli (NASA GSFC 616, SSAI), Graham Trolley (NASA GSFC 616)

[edl]: https://urs.earthdata.nasa.gov/
[oci-data-access]: https://oceancolor.gsfc.nasa.gov/resources/docs/tutorials/notebooks/oci_data_access/

## Summary
This notebook demonstrates an attempt to identify phytoplankton blooms using PACE satellite imagery. The results are summarized in a html file.

### Imports

In [ ]:
# --- Standard library imports ---
import os            # Operating system utilities
import sys           # System-specific parameters and functions
import glob          # Unix style pathname pattern expansion
import shutil        # High-level file operations
from pathlib import Path  # Object-oriented filesystem paths
import argparse      # Command-line argument parsing

# --- Scientific computing and data handling ---
import numpy as np           # Numerical operations on arrays
import xarray as xr          # Labeled multi-dimensional arrays (netCDF, etc.)
import importlib             # Import utilities (for reloading modules)
from scipy.ndimage import label  # Image processing (connected component labeling)

# --- Plotting and visualization ---
import matplotlib.pyplot as plt            # Main plotting library
import matplotlib.patches as patches       # Drawing shapes (e.g., rectangles)
from matplotlib.colors import LogNorm      # Logarithmic normalization for colormaps
from matplotlib import rcParams            # Matplotlib runtime configuration
from mpl_toolkits.axes_grid1 import make_axes_locatable  # Advanced axes layout

# --- Geospatial plotting ---
import cartopy.crs as ccrs        # Cartopy coordinate reference systems
import cartopy.feature as cfeature # Cartopy map features (land, ocean, etc.)
import cmocean                    # Oceanographic colormaps

# --- Web and cloud access ---
import requests      # HTTP requests (downloading files, APIs)
import earthaccess   # NASA Earthdata cloud and data access

# --- Update the path for custom tools ---
util_path = os.path.expanduser(
    '~/Documents/GitHub/pace-rapid-response/PRR_OC/Bloom_Detection/utilities'
)
sys.path.append(util_path)

# --- Custom tool imports (reload for development convenience) ---
import detection_util_MK
# import detection_html_all_MK
# import detection_plot_map_MK
# import detection_download_MK

importlib.reload(detection_util_MK)
# importlib.reload(detection_html_all_MK)
# importlib.reload(detection_plot_map_MK)
# importlib.reload(detection_download_MK)

from detection_util_MK import *
# from detection_html_all_MK import *
# from detection_plot_map_MK import *
# from detection_download_MK import *

# Change default font to something available
#rcParams['font.family'] = 'serif' 
rcParams['font.size'] = '16' 

### Setup system and download L3 data

In [ ]:
auth = earthaccess.login(persist=True)

tspan = ('2025-09-16', '2025-09-16')

data_path, l2_path, l3_path, plot_path, html_path = setup_data(tspan)

filelist_l3_all = download_l3_all_chl(tspan, l3_path, days_prior=30,short_name='PACE_OCI_L3M_CHL_NRT', granule_name='*.DAY.*4km*')


### Calculate 30-day mean, compute Chlorophyll-a anomaly, and identify bounding boxes with anomaly >= 1mg/m^3

In [ ]:
print(filelist_l3_all[-8:])
# Open L3 datasets
l3_ds_target = xr.open_mfdataset(filelist_l3_all[-1], combine='nested', concat_dim='time') # open current day L3 data for anomaly calculation
l3_ds_window = xr.open_mfdataset(filelist_l3_all[0:-1], combine='nested', concat_dim='time') # open previous day L3 data for anomaly calculation

# Identify L3 bounding boxes with chlorophyll-a anomaly greater than 1 mg/m^3
l3_bboxes, l3_bboxes_0360 = l3_anomaly_bbox(l3_ds_target, l3_ds_window)

### Plot and save L3 daily, 30-day mean, and Chlorophyll-a anomaly

In [ ]:
L3_data_plot_chl(l3_ds_target, l3_ds_window, l3_bboxes, plot_path, show_fig=True, figsave=True)

### Locate unique L2 granules corresponding to L3 chlorophyll-a anomaly bounding boxes

In [ ]:
short_names = ['PACE_OCI_L2_SFREFL_NRT', 'PACE_OCI_L2_BGC_NRT', 'PACE_OCI_L2_AOP_NRT']
final_results = l2_granules_by_l3bbox(l3_bboxes, tspan, short_names=short_names, print_flag=False)

### Download or cloud open L2 data

In [ ]:
l2_data_paths = download_open_l2(final_results, data_path=l2_path)

### Filter Data

In [ ]:
l2_data_paths_filt, granule_bbox_pixel_counts = filter_l2_by_valid(l2_data_paths, l3_bboxes_0360)

### Plot and save filtered L2 granules indicating regions of interest from L3 chlorophyll anomaly results

In [ ]:
plot_save_BGC_l2_overlay([l2_data_paths_filt[0]], plot_path, 'chlor_a', l3_bboxes, granule_bbox_pixel_counts, vmin=0.1, vmax=30, show_fig=True, figsave=True)
#plot_save_BGC_l2_overlay(l2_data_paths_filt, plot_path, 'poc', l3_bboxes, granule_bbox_pixel_counts, vmin=1, vmax=1000, show_fig=False, figsave=True)
#plot_save_BGC_l2_overlay(l2_data_paths_filt, plot_path, 'carbon_phyto', l3_bboxes, granule_bbox_pixel_counts, vmin=1, vmax=1000, show_fig=False, figsave=True)

